# Сегментация клиентов | Hypothesis Lab

Этот ноутбук запускается **после подготовки данных**. Загрузите ZIP, который выдал ноутбук подготовки: он содержит `segmentation_matrix.csv` и `client_features_combined.csv`. Здесь алгоритм делит клиентов на группы и создает кандидатов аудиторий для дальнейшего агента гипотез.

Для тестового набора используйте архив из **исправленного** ноутбука подготовки. Если в `run_report.json` нет подтвержденных поведенческих расчетов, об этом появится предупреждение. Продуктовый JSON пока не нужен: он поступает вместе с кандидатами аудиторий на следующем этапе.

Сегменты здесь предварительные. JTBD, боли, потребности и мотивы не становятся фактами из одного только поведения: в выходном JSON они остаются `null`, пока не появятся дополнительные источники или отдельно помеченные гипотезы. `client_id` обезличен, поэтому фактическую доступность клиентов в рекламном канале нужно проверить отдельно.


## 1. Настройки

`MIN_SEGMENTS` и `MAX_SEGMENTS` задают диапазон проверяемых вариантов, а не желаемый ответ. Алгоритм сравнивает их по разделимости, минимальному размеру группы и воспроизводимости при смене случайного начального состояния. Если данных недостаточно для мелких групп, не стоит принудительно увеличивать число сегментов.

`DATA_PROVENANCE="synthetic_demo"` означает, что текущий набор синтетический. Для реальных данных укажите `real_clients` после проверки источника. Для исторического теста **до запуска** укажите `EXPERIMENT_START_DATE`: максимальная дата признаков должна быть раньше начала эксперимента. Если оставить `None`, в отчете будет отмечено, что отсутствие будущих событий не доказано.

Дополнительный `LATER_ARCHIVE_PATH` позволяет проверить изменение назначений клиентов в следующем квартале. Второй архив должен быть подготовлен с `AS_OF_DATE` следующего квартала и с тем же `normalization_params.json` из базового периода. Клики и конверсии после даты среза используются только для отдельной оценки, если когда-нибудь появятся, и **не входят** в матрицу сегментации.


In [ ]:
import os
from pathlib import Path

PREPARED_ARCHIVE_PATH = os.getenv("SEGMENTATION_INPUT_PATH")  # None: выбрать ZIP в Colab
LATER_ARCHIVE_PATH = None  # при проверке по кварталам: путь к архиву следующего периода
EXPERIMENT_START_DATE = None  # пример: "2026-10-01", если эксперимент начинается 1 октября
DATA_PROVENANCE = "synthetic_demo"  # для проверенных реальных данных: "real_clients"
MIN_SEGMENTS = 2
MAX_SEGMENTS = 10
MIN_SHARE = 0.03  # сегмент не меньше 3% клиентов, минимум 10 человек
RANDOM_STATE = 42
OUTPUT_DIR = Path(os.getenv("SEGMENTATION_OUTPUT_DIR", "segmentation_output"))


## 2. Загрузка архива

В Colab выберите ZIP из этапа подготовки. Он читается внутри архива, без распаковки файлов в рабочую папку. При ошибке в ID, матрице или рецепте расчет остановится с объяснением.


In [ ]:
import sys
if not PREPARED_ARCHIVE_PATH:
    if "google.colab" not in sys.modules:
        raise ValueError("Задайте SEGMENTATION_INPUT_PATH или загрузите архив через Colab")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Выберите ровно один ZIP этапа подготовки")
    PREPARED_ARCHIVE_PATH = next(iter(uploaded))
print("Архив подготовки:", Path(PREPARED_ARCHIVE_PATH).name)


## 3. Проверка признаков и выбор числа сегментов

На вход модели идет только `segmentation_matrix.csv`. Группа взаимосвязанных поведенческих расчетов получает общий вес, чтобы десятки сумм по категориям не подавляли остальные признаки только своим количеством. Для каждого числа сегментов рассчитываются silhouette, Davies–Bouldin, Calinski–Harabasz, минимальный размер и согласованность запусков K-means. Другой алгоритм, Ward, используется как дополнительная проверка согласия, а не как «эталонная правда».

После выбора числа групп K-means повторно обучается и проверяется на случайных подвыборках клиентов. Признаки для описания групп берутся из читаемой таблицы `client_features_combined.csv` **после** обучения модели. Важно отличать внутреннюю устойчивость от проверки на следующих кварталах.


In [ ]:
"""Exploratory client segmentation from a verified preparation archive.

The model sees only the prepared numeric matrix. Original client features are
used after fitting to describe clusters; no LLM or product data enter fitting.
"""

from __future__ import annotations

import io
import json
import math
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score,
    silhouette_score,
)


REQUIRED_FILES = {"segmentation_matrix.csv", "client_features_combined.csv",
                  "normalization_params.json", "run_report.json"}


def load_prepared_archive(path, *, allow_constant=False):
    """Read files without extracting an untrusted ZIP to disk."""
    path = Path(path)
    with zipfile.ZipFile(path) as z:
        missing = REQUIRED_FILES - set(z.namelist())
        if missing:
            raise ValueError(f"В архиве нет обязательных файлов: {sorted(missing)}")
        matrix = pd.read_csv(io.BytesIO(z.read("segmentation_matrix.csv")), dtype={"__entity_id": "string"})
        profile = pd.read_csv(io.BytesIO(z.read("client_features_combined.csv")), dtype={"__entity_id": "string"})
        recipe = json.loads(z.read("normalization_params.json"))
        report = json.loads(z.read("run_report.json"))
    for name, frame in (("segmentation_matrix", matrix), ("client_features_combined", profile)):
        if ("__entity_id" not in frame or frame["__entity_id"].isna().any()
                or not frame["__entity_id"].is_unique):
            raise ValueError(f"{name}: требуется уникальный непустой __entity_id")
    if set(matrix["__entity_id"]) != set(profile["__entity_id"]):
        raise ValueError("Идентификаторы матрицы и профиля клиентов расходятся")
    if matrix.shape[0] < 20 or matrix.shape[1] < 3:
        raise ValueError("Недостаточно клиентов или признаков для сегментации")
    cols = list(matrix.columns[1:])
    if recipe.get("output_columns") is not None and cols != recipe["output_columns"]:
        raise ValueError("Столбцы матрицы не совпадают с сохраненным рецептом")
    numeric = matrix.drop(columns="__entity_id")
    if any(not pd.api.types.is_numeric_dtype(numeric[c]) for c in numeric):
        raise ValueError("Матрица содержит текстовые признаки")
    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError("В матрице есть пропуски или бесконечные числа")
    if not allow_constant and numeric.nunique(dropna=False).le(1).any():
        raise ValueError("В матрице есть постоянные столбцы: повторите подготовку данных")
    profile = profile.set_index("__entity_id").loc[matrix["__entity_id"]].reset_index()
    return matrix, profile, recipe, report


def _group_name(column):
    if "__rollup__" in column:
        return column.split("__rollup__", 1)[0] + "__rollup__" + column.split("__rollup__", 1)[1].split("__", 1)[0]
    match = re.match(r"(__cat\d{3}__)", column)
    return match.group(1) if match else column.removesuffix("__scaled").removesuffix("__missing")


def weighted_matrix(matrix):
    """Limit the total weight of one declared behavior family with many fields."""
    names = list(matrix.columns[1:])
    families = [_group_name(c) for c in names]
    counts = pd.Series(families).value_counts().to_dict()
    weights = np.array([1 / math.sqrt(counts[f]) if "__rollup__" in f else 1.0
                        for f in families], dtype=float)
    return matrix[names].to_numpy(dtype=float) * weights, dict(zip(names, weights))


def choose_k(x, min_k=2, max_k=10, min_share=0.03, random_state=42):
    n = len(x)
    min_clients = max(10, math.ceil(n * min_share))
    distinct_profiles = len(np.unique(x, axis=0))
    max_k = min(max_k, n // min_clients, n - 1, distinct_profiles)
    if min_k < 2 or max_k < min_k:
        raise ValueError("Мало клиентов для заданного числа сегментов и минимального размера")
    rows = []
    for k in range(min_k, max_k + 1):
        base = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(x)
        labels = base.labels_
        sizes = np.bincount(labels, minlength=k)
        seed_ari = []
        for seed in (random_state + 1, random_state + 2):
            other = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(x)
            seed_ari.append(adjusted_rand_score(labels, other))
        sil = float(silhouette_score(x, labels, sample_size=min(n, 2000), random_state=random_state))
        db = float(davies_bouldin_score(x, labels))
        ch = float(calinski_harabasz_score(x, labels))
        stability = float(np.mean(seed_ari))
        viable = int(sizes.min()) >= min_clients and sil > 0 and stability >= 0.7
        rows.append({"k": k, "silhouette": round(sil, 4), "davies_bouldin": round(db, 4),
                     "calinski_harabasz": round(ch, 2), "seed_ari": round(stability, 4),
                     "smallest_segment": int(sizes.min()), "largest_segment": int(sizes.max()),
                     "passes_basic_checks": bool(viable)})
    scores = pd.DataFrame(rows)
    pool = scores.loc[scores["passes_basic_checks"]]
    status = "passed_internal_checks"
    if pool.empty:
        pool = scores.loc[scores["smallest_segment"].ge(min_clients)]
        status = "exploratory_no_candidate_passed_all_checks"
    if pool.empty:
        pool = scores
        status = "exploratory_small_segments"
    chosen = pool.sort_values(["silhouette", "seed_ari", "k"], ascending=[False, False, True]).iloc[0]
    return int(chosen["k"]), scores, status, min_clients


def _relabel(labels):
    counts = pd.Series(labels).value_counts()
    order = sorted(counts.index, key=lambda value: (-int(counts[value]), int(value)))
    mapping = {int(old): f"seg_{i:02d}" for i, old in enumerate(order, start=1)}
    return pd.Series(labels).map(mapping).to_numpy(), mapping


def _segment_bootstrap_stability(x, reference, k, random_state=42, repeats=5):
    rng = np.random.default_rng(random_state + 173)
    n = len(x)
    original = sorted(np.unique(reference))
    overlaps = {int(label): [] for label in original}
    ari = []
    for iteration in range(repeats):
        subsample = rng.choice(n, size=max(k * 2, int(0.8 * n)), replace=False)
        model = KMeans(n_clusters=k, n_init=10, random_state=random_state + 100 + iteration).fit(x[subsample])
        predicted = model.predict(x)
        ari.append(float(adjusted_rand_score(reference, predicted)))
        contingency = np.array([[(np.logical_and(reference == old, predicted == new)).sum()
                                 for new in range(k)] for old in original])
        rr, cc = linear_sum_assignment(-contingency)
        for r, c in zip(rr, cc):
            a = reference == original[r]
            b = predicted == c
            overlaps[int(original[r])].append(float(np.logical_and(a, b).sum() / np.logical_or(a, b).sum()))
    return {"bootstrap_mean_ari": round(float(np.mean(ari)), 4),
            "per_original_label_mean_jaccard": {str(label): round(float(np.mean(values)), 4)
                                                for label, values in overlaps.items()},
            "repeats": repeats, "train_fraction": 0.8}


def _category_mapping(recipe):
    mapping = {}
    for i, entry in enumerate(recipe.get("categories", []), start=1):
        group = f"__cat{i:03d}__"
        for j, value in enumerate(entry["levels"], start=1):
            mapping[group + f"value_{j:02d}"] = (entry["field"], value)
        mapping[group + "other"] = (entry["field"], "другие значения")
        mapping[group + "missing"] = (entry["field"], "нет значения")
    return mapping


def _rollup_mapping(report):
    mapping = {}
    for table_no, table in enumerate(report.get("tables", []), start=1):
        for rule in table.get("client_features", {}).get("behavior_rollups_declared", []):
            prefix = f"__t{table_no:02d}__rollup__{rule['name']}__"
            for i, value in enumerate(rule.get("category_values") or [], start=1):
                mapping[prefix + f"value_{i:02d}__"] = str(value)
    return mapping


def _display_name(name, rollup_map):
    for prefix, value in rollup_map.items():
        if name.startswith(prefix):
            return name.replace(prefix, prefix + value + "__", 1)
    return name


def describe_segment(mask, matrix, profile, recipe, report, max_features=8):
    numbers = matrix.drop(columns="__entity_id")
    category_map = _category_mapping(recipe)
    rollup_map = _rollup_mapping(report)
    drivers = []
    for col in numbers:
        all_values = numbers[col].to_numpy(dtype=float)
        subset = all_values[mask]
        if col in category_map:
            field, category = category_map[col]
            # Prep encodes each category with 1/sqrt(2).
            segment_share = float(np.mean(subset > 0))
            overall_share = float(np.mean(all_values > 0))
            effect = (segment_share - overall_share) / max(0.1, math.sqrt(overall_share * (1 - overall_share)))
            drivers.append({"feature": field, "value": category, "kind": "category",
                            "segment_share": round(segment_share, 4), "overall_share": round(overall_share, 4),
                            "direction": "higher" if effect > 0 else "lower", "effect_size": round(abs(effect), 3),
                            "family": _group_name(col)})
        elif col.endswith("__scaled"):
            unscaled = col.removesuffix("__scaled")
            variance = float(np.std(all_values))
            if variance < 1e-10:
                continue
            effect = float((np.mean(subset) - np.mean(all_values)) / variance)
            entry = {"feature": _display_name(unscaled, rollup_map), "kind": "numeric",
                     "direction": "higher" if effect > 0 else "lower", "effect_size": round(abs(effect), 3),
                     "family": _group_name(col)}
            if unscaled in profile:
                raw = pd.to_numeric(profile[unscaled], errors="coerce")
                segment_value = raw[mask].median()
                overall_value = raw.median()
                entry["segment_median"] = float(segment_value) if pd.notna(segment_value) else None
                entry["overall_median"] = float(overall_value) if pd.notna(overall_value) else None
                entry["segment_mean"] = round(float(raw[mask].mean()), 2) if raw[mask].notna().any() else None
                entry["overall_mean"] = round(float(raw.mean()), 2) if raw.notna().any() else None
            else:
                entry["segment_scaled_mean"] = round(float(np.mean(subset)), 3)
                entry["overall_scaled_mean"] = round(float(np.mean(all_values)), 3)
            drivers.append(entry)
    drivers.sort(key=lambda item: item["effect_size"], reverse=True)
    selected = []
    family_counts = {}
    for entry in drivers:
        if entry["effect_size"] < 0.15:
            continue
        family = entry.pop("family")
        if family_counts.get(family, 0) >= 2:
            continue
        family_counts[family] = family_counts.get(family, 0) + 1
        selected.append(entry)
        if len(selected) == max_features:
            break
    return selected


def _as_of_dates(report):
    result = []
    for t in report.get("tables", []):
        info = t.get("client_features", {})
        if info.get("as_of_utc"):
            result.append(info["as_of_utc"])
    return result


def _temporal_check(later_path, baseline_matrix, baseline_recipe, baseline_report, model,
                    feature_weights, mapping):
    if not later_path:
        return {"status": "not_evaluated", "reason": "Не передан архив следующего периода"}
    # A feature can become constant in a later period without invalidating the
    # frozen training recipe; that change is itself possible drift.
    future_matrix, _, future_recipe, future_report = load_prepared_archive(later_path, allow_constant=True)
    if future_matrix.columns.tolist() != baseline_matrix.columns.tolist() or future_recipe != baseline_recipe:
        raise ValueError("Для проверки по периодам пересчитайте второй архив с normalization_params.json первого периода")
    earlier = [pd.Timestamp(x) for x in _as_of_dates(baseline_report)]
    later = [pd.Timestamp(x) for x in _as_of_dates(future_report)]
    if earlier and later and min(later) <= max(earlier):
        raise ValueError("Второй архив должен иметь более позднюю дату среза")
    x2 = future_matrix.drop(columns="__entity_id").to_numpy(dtype=float) * feature_weights
    predicted = model.predict(x2)
    baseline_ids = baseline_matrix["__entity_id"].astype(str).to_numpy()
    later_ids = future_matrix["__entity_id"].astype(str).to_numpy()
    base = pd.Series(model.labels_, index=baseline_ids)
    current = pd.Series(predicted, index=later_ids)
    common = base.index.intersection(current.index)
    if not len(common):
        return {"status": "not_comparable", "reason": "Нет клиентов с одинаковым ID в двух периодах"}
    return {"status": "evaluated", "earlier_as_of": [str(x) for x in earlier],
            "later_as_of": [str(x) for x in later], "shared_clients": int(len(common)),
            "label_ari_on_shared_clients": round(float(adjusted_rand_score(base.loc[common], current.loc[common])), 4),
            "same_segment_share_on_shared_clients": round(float(np.mean(base.loc[common].to_numpy() == current.loc[common].to_numpy())), 4),
            "later_segment_sizes": {mapping[i]: int((predicted == i).sum()) for i in mapping},
            "note": "Проверка дрейфа назначений; без эталонных меток это не точность сегментации"}


def segment_archive(path, later_path=None, *, min_k=2, max_k=10, min_share=0.03,
                    data_provenance="synthetic_demo", experiment_start_date=None, random_state=42):
    matrix, profile, recipe, prep_report = load_prepared_archive(path)
    as_ofs = _as_of_dates(prep_report)
    if experiment_start_date:
        starts = pd.Timestamp(experiment_start_date)
        if starts.tzinfo is None:
            starts = starts.tz_localize("UTC")
        if any(pd.Timestamp(date) >= starts for date in as_ofs):
            raise ValueError("Дата среза должна быть раньше начала эксперимента: есть риск утечки будущих событий")
    x, weight_map = weighted_matrix(matrix)
    k, scores, selection_status, min_clients = choose_k(x, min_k, max_k, min_share, random_state)
    model = KMeans(n_clusters=k, n_init=20, random_state=random_state).fit(x)
    labels, mapping = _relabel(model.labels_)
    stability = _segment_bootstrap_stability(x, model.labels_, k, random_state)
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(x) if len(x) <= 3000 else None
    ward_ari = None if ward is None else round(float(adjusted_rand_score(model.labels_, ward)), 4)
    temporal = _temporal_check(later_path, matrix, recipe, prep_report, model,
                               np.array(list(weight_map.values())), mapping)
    assignment = pd.DataFrame({"client_id": matrix["__entity_id"].astype(str), "segment_id": labels})
    candidates = []
    for old, segment_id in mapping.items():
        mask = model.labels_ == old
        size = int(mask.sum())
        drivers = describe_segment(mask, matrix, profile, recipe, prep_report)
        jaccard = stability["per_original_label_mean_jaccard"][str(old)]
        risks = []
        if jaccard < 0.7:
            risks.append("Группа меняется при переобучении на части клиентов")
        if size < min_clients:
            risks.append("Группа меньше минимального рабочего размера")
        if data_provenance == "synthetic_demo":
            risks.append("Размер и закономерности получены на синтетических данных")
        if temporal["status"] != "evaluated":
            risks.append("Устойчивость между периодами не проверена")
        if experiment_start_date is None:
            risks.append("Дата старта эксперимента не задана: отсутствие будущих событий в признаках не подтверждено")
        risks.append("JTBD, боли и мотивация не наблюдаются напрямую в банковских событиях")
        candidates.append({
            "segment_id": segment_id, "size": size,
            "volume": {"observed_clients": size, "share_of_input": round(size / len(x), 4),
                       "population_estimate": None},
            "key_features": drivers,
            "context": {"status": "observed_behavior_only", "signals": drivers[:5]},
            "jtbd": None, "pains": None, "motivation": None, "needs": None,
            "interpretation_status": "not_observed_in_available_data",
            "stability": {"resample_jaccard": jaccard, "seed_ari_for_chosen_k": float(
                scores.loc[scores.k.eq(k), "seed_ari"].iloc[0]),
                          "temporal_status": temporal["status"]},
            "technical_activatability": {"status": "unverified", "reason":
                "Нужны связь client_id с разрешенным каналом, доступность ID и правила коммуникации"},
            "risks": risks,
        })
    warnings = []
    if not any(t.get("client_features", {}).get("behavior_rollups_declared")
               for t in prep_report.get("tables", [])):
        warnings.append("Вход не содержит подтвержденных поведенческих сумм; проверьте правила подготовки")
    if prep_report.get("segmentation_matrix", {}).get("status") != "ready":
        warnings.append("Входная матрица помечена как предварительная: нужен смысловой контроль признаков")
    if experiment_start_date is None:
        warnings.append("Не указана дата начала эксперимента; автоматический контроль утечки будущих событий не выполнен")
    if ward_ari is not None and ward_ari < 0.5:
        warnings.append("Разные алгоритмы дают существенно разные группы")
    if selection_status != "passed_internal_checks":
        warnings.append("Ни одно число сегментов не прошло все базовые внутренние проверки")
    chosen_silhouette = float(scores.loc[scores.k.eq(k), "silhouette"].iloc[0])
    if chosen_silhouette < 0.2:
        warnings.append("Разделение групп слабое по silhouette; сравните с другими признаками и источниками")
    report = {"status": "exploratory_not_ground_truth", "selected_k": k, "selection_status": selection_status,
              "clients": len(x), "features": x.shape[1], "min_size_required": min_clients,
              "source_as_of": as_ofs, "data_provenance": data_provenance,
              "model": "KMeans with behavior-family weighting", "ward_agreement_ari": ward_ari,
              "chosen_silhouette": chosen_silhouette,
              "bootstrap_stability": stability, "temporal_stability": temporal,
              "external_gold_benchmark": "not_available",
              "future_outcomes_as_features": (
                  "cutoff_checked_semantics_still_require_review" if experiment_start_date
                  else "not_verified_without_experiment_start_date"
              ),
              "warnings": warnings}
    output = {"schema_version": 1, "status": report["status"], "source_archive": Path(path).name,
              "as_of": as_ofs, "data_provenance": data_provenance,
              "segments": candidates, "interpretation_rule":
              "Наблюдаемые признаки подтверждаются данными; мотивы и боли требуют отдельного подтверждения",
              "quality_report": "segmentation_report.json"}
    model_info = {"algorithm": "KMeans", "k": k, "features": list(matrix.columns[1:]),
                  "weights": weight_map, "weighted_centers": model.cluster_centers_.tolist(),
                  "label_mapping": {str(key): value for key, value in mapping.items()},
                  "normalization_recipe": recipe}
    return assignment, scores, output, report, model_info


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

assignments, model_selection, candidates, quality, model_info = segment_archive(
    PREPARED_ARCHIVE_PATH,
    later_path=LATER_ARCHIVE_PATH,
    min_k=MIN_SEGMENTS,
    max_k=MAX_SEGMENTS,
    min_share=MIN_SHARE,
    data_provenance=DATA_PROVENANCE,
    experiment_start_date=EXPERIMENT_START_DATE,
    random_state=RANDOM_STATE,
)
print("Выбрано сегментов:", quality["selected_k"])
display(model_selection)
print("Размеры:", assignments.segment_id.value_counts().sort_index().to_dict())
print("Silhouette:", quality["chosen_silhouette"],
      "| согласие с Ward:", quality["ward_agreement_ari"])
print("Предупреждения:")
for warning in quality["warnings"]:
    print("-", warning)
for candidate in candidates["segments"]:
    print("\n", candidate["segment_id"], "клиентов:", candidate["size"])
    display(pd.DataFrame(candidate["key_features"]).head(6))


## 4. Выход для следующего этапа

Архив результата содержит:

- `client_segments.csv` — назначение `client_id → segment_id`;
- `segment_candidates.json` — размеры и наблюдаемые характеристики групп, поля для JTBD/болей и честные отметки об отсутствии таких данных;
- `model_selection.csv`, `segmentation_report.json` — выбор числа групп, проверки и риски;
- `segmentation_model.json` — веса признаков и центры для технического воспроизведения.

Следующий модуль принимает **паспорт продукта JSON + `segment_candidates.json` + базу знаний** и формирует гипотезы. Этот ноутбук не использует паспорт продукта для выделения сегментов и не принимает будущие конверсии в качестве признаков.


In [ ]:
import json
import zipfile

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assignments.to_csv(OUTPUT_DIR / "client_segments.csv", index=False, encoding="utf-8-sig")
model_selection.to_csv(OUTPUT_DIR / "model_selection.csv", index=False, encoding="utf-8-sig")
for filename, payload in (("segment_candidates.json", candidates),
                          ("segmentation_report.json", quality),
                          ("segmentation_model.json", model_info)):
    (OUTPUT_DIR / filename).write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False),
                                       encoding="utf-8")

archive = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path, path.name)
print("Готово:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive))
